# MaNGA drpall HEALPix collision check
Analyze the *actual* catalog used by MMU (`drpall-v3_1_1.fits` MANGA HDU, optionally filtered by DAPDONE) to find the healpix order that resolves all row-level collisions.

In [4]:
from collections import Counter

import healpy as hp
import numpy as np
from astropy.table import Table

FITS = '/mnt/home/thehir/tmp-catalog-downloads/drpall-v3_1_1.fits'
DAPALL = '/mnt/home/thehir/tmp-catalog-downloads/dapall-v3_1_1-3.1.0.fits'

## Load the MANGA HDU and inspect columns

In [5]:
cat = Table.read(FITS, hdu='MANGA')
print(f'n_rows = {len(cat)}')
ra_candidates = [c for c in cat.colnames if c.lower() in ('objra', 'ifura', 'ra', 'catalog_ra')]
dec_candidates = [c for c in cat.colnames if c.lower() in ('objdec', 'ifudec', 'dec', 'catalog_dec')]
id_candidates = [c for c in cat.colnames if c.lower() in ('mangaid', 'plateifu', 'plate', 'ifudsgn')]
print(f'ra columns: {ra_candidates}')
print(f'dec columns: {dec_candidates}')
print(f'id columns: {id_candidates}')

n_rows = 11273
ra columns: ['objra', 'ifura']
dec columns: ['objdec', 'ifudec']
id columns: ['plate', 'ifudsgn', 'plateifu', 'mangaid']


In [10]:
dap = Table.read(DAPALL)
print(dap.colnames)

['PLATE', 'IFUDESIGN', 'PLATEIFU', 'MANGAID', 'DRPALLINDX', 'MODE', 'DAPTYPE', 'DAPDONE', 'OBJRA', 'OBJDEC', 'IFURA', 'IFUDEC', 'MNGTARG1', 'MNGTARG2', 'MNGTARG3', 'Z', 'LDIST_Z', 'ADIST_Z', 'NSA_Z', 'NSA_ZDIST', 'LDIST_NSA_Z', 'ADIST_NSA_Z', 'NSA_ELPETRO_BA', 'NSA_ELPETRO_PHI', 'NSA_ELPETRO_TH50_R', 'NSA_SERSIC_BA', 'NSA_SERSIC_PHI', 'NSA_SERSIC_TH50', 'NSA_SERSIC_N', 'VERSDRP2', 'VERSDRP3', 'VERSCORE', 'VERSUTIL', 'VERSDAP', 'DRP3QUAL', 'DAPQUAL', 'RDXQAKEY', 'BINKEY', 'SCKEY', 'ELMKEY', 'ELFKEY', 'SIKEY', 'BINTYPE', 'BINSNR', 'TPLKEY', 'DATEDAP', 'DAPBINS', 'RCOV90', 'SNR_MED', 'SNR_RING', 'SB_1RE', 'BIN_RMAX', 'BIN_R_N', 'BIN_R_SNR', 'STELLAR_Z', 'STELLAR_VEL_LO', 'STELLAR_VEL_HI', 'STELLAR_VEL_LO_CLIP', 'STELLAR_VEL_HI_CLIP', 'STELLAR_SIGMA_1RE', 'STELLAR_RCHI2_1RE', 'HA_Z', 'HA_GVEL_LO', 'HA_GVEL_HI', 'HA_GVEL_LO_CLIP', 'HA_GVEL_HI_CLIP', 'HA_GSIGMA_1RE', 'HA_GSIGMA_HI', 'HA_GSIGMA_HI_CLIP', 'EMLINE_RCHI2_1RE', 'EMLINE_SFLUX_CEN', 'EMLINE_SFLUX_1RE', 'EMLINE_SFLUX_TOT', 'EMLINE_S

In [11]:
# dap = Table.read(DAPALL, hdu='DAPALL')
dap = Table.read(DAPALL)
print(f'dapall rows: {len(dap)}')
dap_cols = {c.lower(): c for c in dap.colnames}
print(f'has DAPDONE: {"dapdone" in dap_cols}, has PLATEIFU: {"plateifu" in dap_cols}')

# DAPALL has one row per (plateifu, daptype). Collapse to plateifu-level DAPDONE.
dap_pifu_col = dap_cols['plateifu']
dap_done_col = dap_cols['dapdone']
keep_pifu = set()
for pifu, done in zip(dap[dap_pifu_col], dap[dap_done_col]):
    if not bool(done):
        continue
    p = pifu.decode() if isinstance(pifu, bytes) else str(pifu)
    keep_pifu.add(p)
print(f'plateifu with DAPDONE=True (any daptype): {len(keep_pifu)}')

drp_pifu = np.array([
    (x.decode() if isinstance(x, bytes) else str(x)) for x in cat['plateifu']
])
mask = np.array([p in keep_pifu for p in drp_pifu])
print(f'drpall rows kept: {mask.sum()}/{len(cat)}')
cat = cat[mask]
print(f'cat rows after filter: {len(cat)}')

dapall rows: 10782
has DAPDONE: True, has PLATEIFU: True
plateifu with DAPDONE=True (any daptype): 10737
drpall rows kept: 10737/11273
cat rows after filter: 10737


## Apply MMU's filter: inner-join with dapall on plateifu, keep DAPDONE == True

This mirrors `scripts/manga/build_parent_sample.py` in the MultimodalUniverse repo.

In [12]:
# MMU uses objra/objdec via build_parent_sample.py. Stick with that.
RA_COL = 'objra' if 'objra' in cat.colnames else ra_candidates[0]
DEC_COL = 'objdec' if 'objdec' in cat.colnames else dec_candidates[0]
print(f'using {RA_COL}/{DEC_COL}')

ra = np.asarray(cat[RA_COL], dtype=np.float64)
dec = np.asarray(cat[DEC_COL], dtype=np.float64)

finite = np.isfinite(ra) & np.isfinite(dec)
print(f'finite coords: {finite.sum()}/{len(ra)}')
ra, dec = ra[finite], dec[finite]

using objra/objdec
finite coords: 10737/10737


## How many unique galaxies vs observations?
MaNGA reobserves galaxies — same `mangaid` can produce multiple `plateifu` rows at identical (objra, objdec).

In [13]:
if 'mangaid' in cat.colnames:
    mangaids = [x.decode() if isinstance(x, bytes) else str(x) for x in cat['mangaid'][finite]]
    print(f'unique mangaid: {len(set(mangaids))}')
if 'plateifu' in cat.colnames:
    plateifus = [x.decode() if isinstance(x, bytes) else str(x) for x in cat['plateifu'][finite]]
    print(f'unique plateifu: {len(set(plateifus))}')

coord_counts = Counter(zip(ra.tolist(), dec.tolist()))
dup_coords = [(r, d, n) for (r, d), n in coord_counts.items() if n > 1]
print(f'bit-identical (ra, dec) groups: {len(dup_coords)}')
print(f'total rows in identical groups: {sum(n for _, _, n in dup_coords)}')
print(f'max rows at one coordinate: {max((n for _, _, n in dup_coords), default=0)}')

unique mangaid: 10556
unique plateifu: 10737
bit-identical (ra, dec) groups: 146
total rows in identical groups: 319
max rows at one coordinate: 5


## Healpix collision sweep

In [15]:
for order in range(5, 30):
    nside = 2 ** order
    pix = hp.ang2pix(nside, ra, dec, lonlat=True, nest=True)
    _, counts = np.unique(pix, return_counts=True)
    print(f'order={order:2d} max_per_pixel={counts.max():5d} n_collisions={(counts>1).sum()}')

order= 5 max_per_pixel=  345 n_collisions=1039
order= 6 max_per_pixel=  229 n_collisions=2274
order= 7 max_per_pixel=  226 n_collisions=2274
order= 8 max_per_pixel=  117 n_collisions=1410
order= 9 max_per_pixel=  117 n_collisions=726
order=10 max_per_pixel=   62 n_collisions=376
order=11 max_per_pixel=   22 n_collisions=236
order=12 max_per_pixel=    6 n_collisions=257
order=13 max_per_pixel=    5 n_collisions=214
order=14 max_per_pixel=    5 n_collisions=159
order=15 max_per_pixel=    5 n_collisions=155
order=16 max_per_pixel=    5 n_collisions=153
order=17 max_per_pixel=    5 n_collisions=152
order=18 max_per_pixel=    5 n_collisions=150
order=19 max_per_pixel=    5 n_collisions=149
order=20 max_per_pixel=    5 n_collisions=148
order=21 max_per_pixel=    5 n_collisions=147
order=22 max_per_pixel=    5 n_collisions=145
order=23 max_per_pixel=    5 n_collisions=145
order=24 max_per_pixel=    5 n_collisions=146
order=25 max_per_pixel=    5 n_collisions=146
order=26 max_per_pixel=    5 n

## Inspect the bit-identical duplicates
If any remain at infinite order, list their mangaid/plateifu to see whether reobservations explain them.

In [16]:
ra_all = np.asarray(cat[RA_COL], dtype=np.float64)
dec_all = np.asarray(cat[DEC_COL], dtype=np.float64)
mangaid_all = cat['mangaid'] if 'mangaid' in cat.colnames else None
plateifu_all = cat['plateifu'] if 'plateifu' in cat.colnames else None

for r, d, n in dup_coords:
    idx = np.where((ra_all == r) & (dec_all == d))[0]
    print(f'\nra={r:.6f} dec={d:.6f} count={n} rows={list(idx)}')
    for i in idx:
        mid = mangaid_all[i] if mangaid_all is not None else ''
        pid = plateifu_all[i] if plateifu_all is not None else ''
        if isinstance(mid, bytes): mid = mid.decode()
        if isinstance(pid, bytes): pid = pid.decode()
        print(f'  row={i} mangaid={mid!r} plateifu={pid!r}')


ra=56.702338 dec=68.096625 count=2 rows=[np.int64(17), np.int64(10152)]
  row=17 mangaid='51-100' plateifu='10141-12701'
  row=10152 mangaid='51-49' plateifu='9675-12703'

ra=119.851750 dec=16.418944 count=2 rows=[np.int64(157), np.int64(1432)]
  row=157 mangaid='59-80' plateifu='10217-12703'
  row=1432 mangaid='54-80' plateifu='11744-3702'

ra=140.065052 dec=1.288840 count=2 rows=[np.int64(766), np.int64(10097)]
  row=766 mangaid='1-53985' plateifu='10513-1901'
  row=10097 mangaid='1-53985' plateifu='9512-6104'

ra=139.925621 dec=1.325809 count=2 rows=[np.int64(769), np.int64(10096)]
  row=769 mangaid='1-53987' plateifu='10513-3702'
  row=10096 mangaid='1-53987' plateifu='9512-6103'

ra=149.707718 dec=0.836658 count=2 rows=[np.int64(1002), np.int64(1984)]
  row=1002 mangaid='1-1033' plateifu='10843-12704'
  row=1984 mangaid='1-1033' plateifu='11866-9101'

ra=148.846633 dec=0.878934 count=2 rows=[np.int64(1012), np.int64(1974)]
  row=1012 mangaid='1-908' plateifu='10843-6103'
  row=19

## Cross-check against HDF5 histogram
The staging histogram has 10,735 rows at order 10. Does filtering drpall reproduce that count?

In [17]:
HIST = '/mnt/ceph/users/polymathic/manga-staging-tmp/mmu_manga/intermediate/mmu_manga/intermediate/row_count_mapping_histogram.npz'
with open(HIST, 'rb') as f:
    full = np.frombuffer(f.read(), dtype=np.int64)
print(f'hdf5 rows (histogram): {full.sum()}')
print(f'drpall MANGA rows: {len(cat)}')
print(f'drpall MANGA rows w/ finite coords: {finite.sum()}')
print(f'diff: drpall - hdf5 = {finite.sum() - full.sum()}')

hdf5 rows (histogram): 10735
drpall MANGA rows: 10737
drpall MANGA rows w/ finite coords: 10737
diff: drpall - hdf5 = 2
